# 07 Route Geography

## Purpose

This notebook links Houston METRO route identifiers to their geographic route shapes using GTFS data.

Ridership data tells us **how many people use a route**, but it does not show **where that service operates**. GTFS shape data provides the latitude and longitude points needed to map each route corridor. This step is necessary before comparing transit service with population density and demographic patterns.

## Inputs

- `data/raw/metro_gtfs/merged/routes.txt`
- `data/raw/metro_gtfs/merged/trips.txt`
- `data/raw/metro_gtfs/merged/shapes.txt`

## Output

- `data/processed/key_route_geometry.csv`

## Why this matters

Transit investment decisions are geographic. Before evaluating whether Houston's transit resources align with population density and demand, the project must first connect high-ridership routes to their actual physical alignments.


## 1. Import libraries and load GTFS files

GTFS is organized like a relational database. This notebook uses three GTFS tables:

- `routes.txt`: route IDs, route names, and route types
- `trips.txt`: scheduled trips and their associated route/shape IDs
- `shapes.txt`: ordered latitude/longitude points used to draw route lines


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

routes = pd.read_csv("../data/raw/metro_gtfs/merged/routes.txt")
trips = pd.read_csv("../data/raw/metro_gtfs/merged/trips.txt")
shapes = pd.read_csv("../data/raw/metro_gtfs/merged/shapes.txt")

print("routes:", routes.shape)
print("trips:", trips.shape)
print("shapes:", shapes.shape)


## 2. Identify key transit corridors

This analysis focuses on five major Houston bus corridors that were identified during the ridership analysis:

- Route 82 — Westheimer
- Route 4 — Beechnut
- Route 46 — Gessner
- Route 2 — Bellaire
- Route 25 — Richmond

These routes were selected because they represent high-ridership corridors and provide a useful starting point for evaluating transit investment opportunities.


In [ ]:
key_routes = [82, 4, 46, 2, 25]

route_shapes = trips[
    trips["route_id"].isin(key_routes)
][
    ["route_id", "shape_id"]
].drop_duplicates()

route_shapes.head(20)


## 3. Check how many shape variations exist per route

A single route can have multiple shape IDs because buses may operate in different directions, branches, or service patterns. This step checks how many shape variations exist for each selected route.


In [ ]:
route_shapes.groupby("route_id")["shape_id"].nunique()


## 4. Select one representative shape per route

For this first version of the project, one representative shape is selected for each route. This creates a simplified corridor-level map that is easier to interpret.

In a future version, the analysis could include all route shape variations or separate inbound and outbound directions.


In [ ]:
primary_shapes = (
    route_shapes
    .groupby("route_id")
    .first()
    .reset_index()
)

primary_shapes


## 5. Add route names to the selected shapes

The selected shape IDs are joined back to the route table so that each geometry can be labeled with its route name.


In [ ]:
primary_shapes = primary_shapes.merge(
    routes[["route_id", "route_short_name", "route_long_name"]],
    on="route_id",
    how="left"
)

primary_shapes


## 6. Build the route geometry dataset

The representative route shapes are merged with the `shapes.txt` table. This creates a table of ordered latitude/longitude points for each selected corridor.


In [ ]:
route_geometry = primary_shapes.merge(
    shapes,
    on="shape_id",
    how="left"
)

route_geometry.head()


## 7. Visualize Route 82 Westheimer

Westheimer is one of the most important corridors in this project because it had the highest weekday ridership among the selected routes. This quick plot verifies that the GTFS shape data can be used to draw the corridor.


In [ ]:
westheimer_shape_id = primary_shapes.loc[
    primary_shapes["route_id"] == 82,
    "shape_id"
].iloc[0]

westheimer = shapes[
    shapes["shape_id"] == westheimer_shape_id
].sort_values("shape_pt_sequence")

plt.figure(figsize=(10, 6))
plt.plot(
    westheimer["shape_pt_lon"],
    westheimer["shape_pt_lat"]
)

plt.title("Route 82 Westheimer - GTFS Shape")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


## 8. Visualize all selected high-ridership routes

This plot shows the selected corridors together. It provides an early geographic view of how major high-ridership bus routes are distributed across Houston.


In [ ]:
plt.figure(figsize=(10, 8))

for _, row in primary_shapes.iterrows():
    shape_id = row["shape_id"]
    route_name = row["route_long_name"]

    route_line = shapes[
        shapes["shape_id"] == shape_id
    ].sort_values("shape_pt_sequence")

    plt.plot(
        route_line["shape_pt_lon"],
        route_line["shape_pt_lat"],
        label=f"{row['route_id']} {route_name}"
    )

plt.title("Selected High-Ridership METRO Routes")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.show()


## 9. Export processed route geometry

The final route geometry dataset is saved for use in later notebooks. This file becomes the route overlay used in the population density and final recommendation maps.


In [ ]:
route_geometry.to_csv(
    "../data/processed/key_route_geometry.csv",
    index=False
)

print("Saved key_route_geometry.csv")


## Summary

This notebook successfully connected selected METRO route IDs to their geographic route shapes.

The resulting file, `key_route_geometry.csv`, allows later notebooks to overlay major transit corridors on top of Census population density data. This is an important step toward answering the project's central question: whether Houston's transit resources align with areas of high demand and dense population.
